In [ ]:
from pathlib import Path

import pandas as pd

In [ ]:
# --------------------------------------------------
# Directories
# --------------------------------------------------
DIR_DATA = Path("data")
DIR_METADATA = DIR_DATA / "0_metadata"
DIR_PROCESSED = DIR_DATA / "2_processed"
DIR_RESULTS = DIR_DATA / "3_results"
DIR_EVAL = DIR_DATA / "4_evaluation"

DIR_ICEYE = DIR_PROCESSED / "1_ICEYE"
DIR_NDVI = DIR_PROCESSED / "2_Sentinel2"
DIR_DEPMAP = DIR_PROCESSED / "3_Terrain" / "depmap"
DIR_ZSCORE = DIR_PROCESSED / "4_ICEYE_Zscore"
DIR_GRADCAM_CLS = DIR_RESULTS / "siamese-gradcam-classification"
DIR_GRADCAM_REG = DIR_RESULTS / "siamese-gradcam-regression"
DIR_ELORA_AREAS = DIR_DATA / "1_org" / "a_mask"

# --------------------------------------------------
# Files
# --------------------------------------------------
FILEPATH_PAIR_MANIFEST = DIR_METADATA / "pair-manifest.csv"
FILEPATH_PATCH_CENTERS = DIR_METADATA / "patch-centers.csv"

THRESHOLD_CLS = 0.5
THRESHOLD_REG = 10
THRESHOLD_GRADCAM = 0.5
MIN_OVERLAP_RATIO = 0.05

FILEPATH_EVAL_SUMMARY = DIR_EVAL / "evaluation-summary-controlled.csv"

# --------------------------------------------------

In [ ]:
def find_gradcam_file(directory, pair_id, patch_id):
    pattern = f"gradcam_{pair_id}_{patch_id}_*.tif"
    matches = sorted(directory.glob(pattern))
    if len(matches) != 1:
        raise FileNotFoundError(
            f"{pattern}: expected 1 file, found {len(matches)}"
        )
    return matches[0]

In [ ]:
# --------------------------------------------------
# Read manifests
# --------------------------------------------------
pair_manifest = pd.read_csv(FILEPATH_PAIR_MANIFEST)

pair_manifest = pair_manifest[
    pair_manifest["split"] == "test"
]

patch_manifest = pd.read_csv(FILEPATH_PATCH_CENTERS)

# --------------------------------------------------
# Test patches only
# --------------------------------------------------
patch_manifest = patch_manifest[
    patch_manifest["usage"] == "test"
]

rows = []

# ==================================================
# Loop over pairs
# ==================================================
for _, pair in pair_manifest.iterrows():

    pair_id = pair["pair_id"]
    year = pair["sar_target"][:4]

    sar_base_dir = DIR_ICEYE / pair["sar_base"]
    sar_target_dir = DIR_ICEYE / pair["sar_target"]

    ndvi_base_dir = (
        DIR_NDVI
        / f"patches_{year}"
        / pair["ndvi_base"]
    )

    ndvi_target_dir = (
        DIR_NDVI
        / f"patches_{year}"
        / pair["ndvi_target"]
    )

    depmap_dir = (
        DIR_DEPMAP
        / f"patches_{year}"
    )

    zscore_dir = (
        DIR_ZSCORE
        / pair["sar_target"]
    )

    # ICEYE image ids
    base_image_id = pair["sar_base"].split("_")[-1]
    target_image_id = pair["sar_target"].split("_")[-1]

    # ----------------------------------------------
    # Loop over all test patches
    # ----------------------------------------------
    for _, patch in patch_manifest.iterrows():

        patch_id = patch["id"]

        # ------------------------------------------
        # Generate BOTH evaluation conditions
        # ------------------------------------------
        for controlled in (False, True):

            suffix = "_controlled" if controlled else ""

            # --------------------------------------
            # SAR
            # --------------------------------------
            sar_base_file = (
                sar_base_dir
                / f"{base_image_id}_{patch_id}.tif"
            )

            sar_target_file = (
                sar_target_dir
                / f"{target_image_id}_{patch_id}{suffix}.tif"
            )

            # Fallback to original if controlled version
            # does not exist.
            if controlled and not sar_target_file.exists():
                sar_target_file = (
                    sar_target_dir
                    / f"{target_image_id}_{patch_id}.tif"
                )

            # --------------------------------------
            # NDVI
            # --------------------------------------
            ndvi_base_file = (
                ndvi_base_dir
                / f"{pair['ndvi_base']}_{patch_id}_NDVI.tif"
            )

            ndvi_target_file = (
                ndvi_target_dir
                / f"{pair['ndvi_target']}_{patch_id}_NDVI{suffix}.tif"
            )

            if controlled and not ndvi_target_file.exists():
                ndvi_target_file = (
                    ndvi_target_dir
                    / f"{pair['ndvi_target']}_{patch_id}_NDVI.tif"
                )

            # --------------------------------------
            # Other files
            # --------------------------------------
            depmap_file = (
                depmap_dir
                / f"depmap_{patch_id}.tif"
            )

            zscore_file = (
                zscore_dir
                / f"z_{target_image_id}_{patch_id}.tif"
            )

            if controlled:
                gradcam_cls_file = find_gradcam_file(
                    DIR_GRADCAM_CLS / "patches-gradcam-controlled",
                    pair_id,
                    patch_id,
                )
            else:
                gradcam_cls_file = find_gradcam_file(
                    DIR_GRADCAM_CLS / "patches-gradcam",
                    pair_id,
                    patch_id,
                )

            if controlled:
                gradcam_reg_file = find_gradcam_file(
                    DIR_GRADCAM_REG / "patches-gradcam-controlled",
                    pair_id,
                    patch_id,
                )
            else:
                gradcam_reg_file = find_gradcam_file(
                    DIR_GRADCAM_REG / "patches-gradcam",
                    pair_id,
                    patch_id,
                )

            # --------------------------------------
            # Skip incomplete records
            # --------------------------------------
            required_files = [
                sar_base_file,
                sar_target_file,
                ndvi_base_file,
                ndvi_target_file,
                depmap_file,
                zscore_file,
                gradcam_cls_file,
                gradcam_reg_file,
            ]

            missing_files = [
                p for p in required_files
                if not p.exists()
            ]

            if not all(p.exists() for p in required_files):
                continue

            area_id = patch["elora_area_id"]
            elora_area = (
                None
                if pd.isna(area_id)
                else DIR_ELORA_AREAS / str(area_id) / f"{area_id}.shp"
            )

            rows.append(
                {
                    "pair_id": pair_id,
                    "patch_id": patch_id,

                    # Evaluation condition
                    "controlled": controlled,

                    # Metadata
                    "elora_area_id": patch["elora_area_id"],
                    "elora_area": elora_area,

                    # Input images
                    "sar_base": sar_base_file,
                    "sar_target": sar_target_file,

                    "ndvi_base": ndvi_base_file,
                    "ndvi_target": ndvi_target_file,

                    # Auxiliary data
                    "depmap": depmap_file,
                    "zscore": zscore_file,

                    # Explanations
                    "gradcam_cls": gradcam_cls_file,
                    "gradcam_reg": gradcam_reg_file,
                }
            )

# --------------------------------------------------
# Result
# --------------------------------------------------
groups_patch_paths = pd.DataFrame(rows)

groups_patch_paths = groups_patch_paths.sort_values(
    ["pair_id", "patch_id", "controlled"]
).reset_index(drop=True)

print(
    groups_patch_paths.groupby(
        ["pair_id", "controlled"]
    ).size()
)

print(groups_patch_paths.head())

print(f"\n{len(groups_patch_paths)} patch records found.")

In [ ]:
groups_patch_paths.to_csv(
    DIR_EVAL / "groups_patch_paths.csv",
    index=False,
)

In [ ]:
import geopandas as gpd
import numpy as np
import rasterio
from affine import Affine
from pydantic import BaseModel
from rasterio.features import rasterize

In [ ]:
class GradCAMOverlapResult(BaseModel):
    overlap: bool
    overlap_ratio: float
    coverage_ratio: float


def gradcam_overlaps_polygon(
    gradcam: np.ndarray,
    polygon_path: Path | None,
    transform: Affine,
    threshold: float = 0.5,
    min_overlap_ratio: float = 0.05,
) -> GradCAMOverlapResult:
    """
    Evaluate whether the highlighted Grad-CAM region overlaps the
    ground-truth flooded area.

    Parameters
    ----------
    gradcam : np.ndarray
        Grad-CAM heatmap normalized to [0, 1].

    polygon_path : Path | None
        Flood polygon shapefile. If None, the entire patch is treated as the
        ground-truth area.

    transform : affine.Affine
        Affine transform of the Grad-CAM raster.

    threshold : float, default=0.5
        Grad-CAM threshold.

    min_overlap_ratio : float, default=0.05
        Minimum required overlap ratio for success.

    Returns
    -------
    GradCAMOverlapResult
    """
    gradcam_mask = gradcam >= threshold

    if polygon_path is None:
        polygon_mask = np.ones_like(gradcam_mask, dtype=bool)
    else:
        gdf = gpd.read_file(polygon_path)

        if gdf.empty:
            return GradCAMOverlapResult(
                overlap=False,
                overlap_ratio=0.0,
                coverage_ratio=0.0,
            )

        polygon_mask = rasterize(
            [(geom, 1) for geom in gdf.geometry],
            out_shape=gradcam.shape,
            transform=transform,
            fill=0,
            dtype=np.uint8,
        ).astype(bool)

    highlighted_pixels = np.count_nonzero(gradcam_mask)

    if highlighted_pixels == 0:
        return GradCAMOverlapResult(
            overlap=False,
            overlap_ratio=0.0,
            coverage_ratio=0.0,
        )

    # ------------------------------------------
    # Compute overlap
    # ------------------------------------------
    overlap_mask = gradcam_mask & polygon_mask
    overlap_pixels = np.count_nonzero(overlap_mask)
    polygon_pixels = np.count_nonzero(polygon_mask)
    overlap_ratio = overlap_pixels / highlighted_pixels
    coverage_ratio = (
        overlap_pixels / polygon_pixels
        if polygon_pixels > 0
        else 0.0
    )
    return GradCAMOverlapResult(
        overlap=overlap_ratio >= min_overlap_ratio,
        overlap_ratio=overlap_ratio,
        coverage_ratio=coverage_ratio,
    )

In [ ]:
results = []

for _, row in groups_patch_paths.iterrows():

    # ------------------------------
    # Read Grad-CAM
    # ------------------------------
    with rasterio.open(row["gradcam_cls"]) as src:
        gradcam_cls = src.read(1)
        transform = src.transform

    # ------------------------------
    # Evaluate Grad-CAM overlap
    # ------------------------------
    if row["controlled"]:
        overlap_result = gradcam_overlaps_polygon(
            gradcam=gradcam_cls,
            polygon_path=row["elora_area"],
            transform=transform,
            threshold=THRESHOLD_GRADCAM,
            min_overlap_ratio=MIN_OVERLAP_RATIO,
        )
    else:
        overlap_result = GradCAMOverlapResult(
            overlap=False,
            overlap_ratio=0.0,
            coverage_ratio=0.0,
        )

    cls_prob = float(
        row["gradcam_cls"]
        .stem
        .split("_prob")[-1]
        .replace("_controlled", "")
    )

    reg_pred = float(
        row["gradcam_reg"]
        .stem
        .split("_pred")[-1]
        .replace("_controlled", "")
    )

    # ------------------------------
    # Store results
    # ------------------------------
    results.append(
        {
            "pair_id": row["pair_id"],
            "patch_id": row["patch_id"],
            "controlled": row["controlled"],
            "cls_prob": cls_prob,
            "reg_pred": reg_pred,
            **overlap_result.model_dump(),
        }
    )

eval_summary = pd.DataFrame(results)
display(eval_summary.head())

In [ ]:
eval_summary.to_csv(
    FILEPATH_EVAL_SUMMARY,
    index=False,
)

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)

In [ ]:
# --------------------------------------------------
# Classification confusion matrix
# --------------------------------------------------
# Use only normal samples
plot_data = eval_summary[
    ~eval_summary["controlled"]
].copy()

# Ground truth
plot_data["gt"] = (
    plot_data["pair_id"]
    .str
    .startswith("W")
)

# Prediction
plot_data["pred"] = (
    plot_data["cls_prob"] >= THRESHOLD_CLS
)

# --------------------------------------------------
# Confusion matrix
# --------------------------------------------------
cm = confusion_matrix(
    plot_data["gt"],
    plot_data["pred"],
    labels=[False, True],
)

print(cm)

In [ ]:
# --------------------------------------------------
# Regression confusion matrix
# --------------------------------------------------
# Use only normal samples
plot_data = eval_summary[
    ~eval_summary["controlled"]
].copy()

# Ground truth
plot_data["gt"] = (
    plot_data["pair_id"]
    .str
    .startswith("W")
)

# Prediction from regression
plot_data["pred"] = (
    plot_data["reg_pred"] >= THRESHOLD_REG
)

# --------------------------------------------------
# Confusion matrix
# --------------------------------------------------
cm = confusion_matrix(
    plot_data["gt"],
    plot_data["pred"],
    labels=[False, True],
)

print(cm)

In [ ]:
# --------------------------------------------------
# Prepare data
# --------------------------------------------------
plot_data = eval_summary[
    ~eval_summary["controlled"]
].copy()

# Ground truth
y_true = (
    plot_data["pair_id"]
    .str
    .startswith("W")
)

# Predictions
y_pred_cls = (
    plot_data["cls_prob"] >= THRESHOLD_CLS
)

y_pred_reg = (
    plot_data["reg_pred"] >= THRESHOLD_REG
)

# --------------------------------------------------
# Metric function
# --------------------------------------------------
def compute_metrics(y_true, y_pred):
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred),
        "Recall": recall_score(y_true, y_pred),
        "F1-score": f1_score(y_true, y_pred),
    }

# --------------------------------------------------
# Create summary table
# --------------------------------------------------
table3 = pd.DataFrame(
    [
        {
            "Model": "Classification",
            **compute_metrics(y_true, y_pred_cls),
        },
        {
            "Model": "Regression",
            **compute_metrics(y_true, y_pred_reg),
        },
    ]
)

# Display percentages
table3_display = table3.copy()

metric_columns = [
    "Accuracy",
    "Precision",
    "Recall",
    "F1-score",
]

table3_display[metric_columns] = (
    table3_display[metric_columns] * 100
).round(1)

display(table3_display)

# Table 1: Confusion matrix of classification (wet/dry)

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

In [ ]:
# --------------------------------------------------
# Classification confusion matrix
# --------------------------------------------------
# Use only normal samples
plot_data = eval_summary[
    eval_summary["controlled"]
].copy()

# Ground truth
plot_data["gt"] = (
    plot_data["pair_id"]
    .str
    .startswith("W")
)

# Prediction
plot_data["pred"] = (
    plot_data["cls_prob"] >= THRESHOLD_CLS
)

# --------------------------------------------------
# Confusion matrix
# --------------------------------------------------
cm = confusion_matrix(
    plot_data["gt"],
    plot_data["pred"],
    labels=[False, True],
)

print(cm)

# Table 2: Confusion matrix of regression-derived wet/dry (using rainfall threshold)

In [ ]:
# --------------------------------------------------
# Regression confusion matrix
# --------------------------------------------------
# Use only normal samples
plot_data = eval_summary[
    eval_summary["controlled"]
].copy()

# Ground truth
plot_data["gt"] = (
    plot_data["pair_id"]
    .str
    .startswith("W")
)

# Prediction from regression
plot_data["pred"] = (
    plot_data["reg_pred"] >= THRESHOLD_REG
)

# --------------------------------------------------
# Confusion matrix
# --------------------------------------------------
cm = confusion_matrix(
    plot_data["gt"],
    plot_data["pred"],
    labels=[False, True],
)

print(cm)

# Table 3: Accuracy, Precision, Recall, F1-score for both classification and regression-derived classification

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)

In [ ]:
# --------------------------------------------------
# Prepare data
# --------------------------------------------------
plot_data = eval_summary[
    eval_summary["controlled"]
].copy()

# Ground truth
y_true = (
    plot_data["pair_id"]
    .str
    .startswith("W")
)

# Predictions
y_pred_cls = (
    plot_data["cls_prob"] >= THRESHOLD_CLS
)

y_pred_reg = (
    plot_data["reg_pred"] >= THRESHOLD_REG
)

# --------------------------------------------------
# Metric function
# --------------------------------------------------
def compute_metrics(y_true, y_pred):
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred),
        "Recall": recall_score(y_true, y_pred),
        "F1-score": f1_score(y_true, y_pred),
    }

# --------------------------------------------------
# Create summary table
# --------------------------------------------------
table3 = pd.DataFrame(
    [
        {
            "Model": "Classification",
            **compute_metrics(y_true, y_pred_cls),
        },
        {
            "Model": "Regression",
            **compute_metrics(y_true, y_pred_reg),
        },
    ]
)

# Display percentages
table3_display = table3.copy()

metric_columns = [
    "Accuracy",
    "Precision",
    "Recall",
    "F1-score",
]

table3_display[metric_columns] = (
    table3_display[metric_columns] * 100
).round(1)

display(table3_display)

# --------------------------------------------------
# Save
# --------------------------------------------------
output_path = (
    DIR_EVAL /
    "table3_classification_metrics.csv"
)

table3_display.to_csv(
    output_path,
    index=False,
)

print(f"Saved: {output_path}")

# Table 4: MAE, RMSE, Spearman’s ρ for regression
→ Refer to na_22_siamese_gradcam_regression_api_mlp.ipynb

# Figure 1: Scatter plot of classification probability vs. regression prediction

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from scipy.stats import spearmanr

In [ ]:
# --------------------------------------------------
# Colors for land types
# --------------------------------------------------
LAND_COLORS = {
    "SOIL": "peru",  # or "saddlebrown"
    "BUILDING": "gray",
    "WATER": "dodgerblue",
    "VEGETATION": "forestgreen",
}

# --------------------------------------------------
# Function
# --------------------------------------------------
def plot_cls_vs_reg(
    data: pd.DataFrame,
    title: str,
    filename: str,
):
    """
    Scatter plot of classification probability vs.
    regression prediction.
    """

    # ---------------------------------------
    # Prepare data
    # ---------------------------------------
    plot_data = data[
        [
            "patch_id",
            "cls_prob",
            "reg_pred",
        ]
    ].dropna().copy()

    # ---------------------------------------
    # Land type from patch ID
    # ---------------------------------------
    land_type_map = {
        "S": "SOIL",
        "B": "BUILDING",
        "V": "VEGETATION",
        "W": "WATER",
    }

    plot_data["land_type"] = (
        plot_data["patch_id"]
        .str[1]
        .str.upper()
        .map(land_type_map)
    )

    # ---------------------------------------
    # Spearman correlation
    # ---------------------------------------
    rho, p_value = spearmanr(
        plot_data["cls_prob"],
        plot_data["reg_pred"],
    )

    print(f"\n{title}")
    print(f"Spearman rho: {rho:.3f}")
    print(f"p-value     : {p_value:.4f}")

    # ---------------------------------------
    # Plot
    # ---------------------------------------
    fig, ax = plt.subplots(
        figsize=(7, 6)
    )

    # ---------------------------------------
    # Scatter by land type
    # ---------------------------------------
    for land_type, color in LAND_COLORS.items():

        subset = plot_data[
            plot_data["land_type"] == land_type
        ]

        if subset.empty:
            continue

        ax.scatter(
            subset["cls_prob"],
            subset["reg_pred"],
            color=color,
            s=60,
            alpha=0.8,
            label=land_type.title(),
        )

    # ---------------------------------------
    # Patch IDs
    # ---------------------------------------
    for _, row in plot_data.iterrows():

        ax.text(
            row["cls_prob"],
            row["reg_pred"],
            row["patch_id"],
            fontsize=7,
        )

    # ---------------------------------------
    # Threshold lines
    # ---------------------------------------
    ax.axvline(
        THRESHOLD_CLS,
        color="black",
        linestyle="--",
        linewidth=1,
    )

    ax.axhline(
        THRESHOLD_REG,
        color="black",
        linestyle="--",
        linewidth=1,
    )

    # ---------------------------------------
    # Labels
    # ---------------------------------------
    ax.set_xlabel(
        "Classification Probability"
    )

    ax.set_ylabel(
        "Predicted Precipitation (mm)"
    )

    ax.set_xlim(0.0, 1.1)
    ax.set_ylim(0.0, 30.0)

    ax.grid(alpha=0.3)

    ax.legend(
        title="Land Type",
        loc="upper left",
    )

    ax.set_title(
        f"{title}\n"
        f"Spearman $\\rho$ = {rho:.3f}, "
        f"$p$ = {p_value:.3g}"
    )

    plt.tight_layout()

    # ---------------------------------------
    # Save
    # ---------------------------------------
    output_path = DIR_EVAL / filename

    fig.savefig(
        output_path,
        dpi=300,
        bbox_inches="tight",
    )

    print(f"Saved: {output_path}")

    plt.show()
    plt.close(fig)


# ---------------------------------------
# Normal Dry
# ---------------------------------------
plot_cls_vs_reg(
    eval_summary[
        (~eval_summary["controlled"])
        & (eval_summary["pair_id"].str.startswith("D"))
    ],
    title="Normal Dry",
    filename="fig_cls_vs_reg_normal_dry.png",
)

# ---------------------------------------
# Normal Wet
# ---------------------------------------
plot_cls_vs_reg(
    eval_summary[
        (~eval_summary["controlled"])
        & (eval_summary["pair_id"].str.startswith("W"))
    ],
    title="Normal Wet",
    filename="fig_cls_vs_reg_normal_wet.png",
)

# ---------------------------------------
# Controlled Wet
# ---------------------------------------
plot_cls_vs_reg(
    eval_summary[
        (eval_summary["controlled"])
        & (eval_summary["pair_id"].str.startswith("W"))
    ],
    title="Controlled Wet",
    filename="fig_cls_vs_reg_controlled_wet.png",
)

# Table 5: Flood detection results of the classification model

In [ ]:
# --------------------------------------------------
# Controlled samples only
# --------------------------------------------------
plot_data = eval_summary[
    eval_summary["controlled"]
].copy()

# --------------------------------------------------
# Evaluate flood detection
# --------------------------------------------------
results = []

for _, row in plot_data.iterrows():

    actual_wet = row["pair_id"].startswith("W")
    pred_wet = row["cls_prob"] >= THRESHOLD_CLS

    if actual_wet:

        # A wet sample is correctly detected only if
        # it is predicted wet AND Grad-CAM overlaps.
        if pred_wet and row["overlap"]:
            result = "TP"
        else:
            result = "FN"

    else:

        # Dry sample
        if pred_wet:
            result = "FP"
        else:
            result = "TN"

    results.append(result)

plot_data["result"] = results

# --------------------------------------------------
# Confusion matrix
# --------------------------------------------------
tp = (plot_data["result"] == "TP").sum()
fp = (plot_data["result"] == "FP").sum()
tn = (plot_data["result"] == "TN").sum()
fn = (plot_data["result"] == "FN").sum()

table5 = pd.DataFrame(
    {
        "Predicted Dry": [
            tn,
            fn,
        ],
        "Predicted Wet": [
            fp,
            tp,
        ],
    },
    index=[
        "Actual Dry",
        "Actual Wet",
    ],
)

display(table5)

# --------------------------------------------------
# Save
# --------------------------------------------------
output_path = (
    DIR_EVAL /
    "table5_flood_detection_classification.csv"
)

table5.to_csv(output_path)

print(f"Saved: {output_path}")

# Table 6: Flood detection results of the regression model

In [ ]:
# --------------------------------------------------
# Controlled samples only
# --------------------------------------------------
plot_data = eval_summary[
    eval_summary["controlled"]
].copy()

# --------------------------------------------------
# Evaluate flood detection
# --------------------------------------------------
results = []

for _, row in plot_data.iterrows():

    actual_wet = row["pair_id"].startswith("W")
    pred_wet = row["reg_pred"] >= THRESHOLD_REG

    if actual_wet:

        # A wet sample is correctly detected only if
        # it is predicted wet AND Grad-CAM overlaps.
        if pred_wet and row["overlap"]:
            result = "TP"
        else:
            result = "FN"

    else:

        # Dry sample
        if pred_wet:
            result = "FP"
        else:
            result = "TN"

    results.append(result)

plot_data["result"] = results

# --------------------------------------------------
# Confusion matrix
# --------------------------------------------------
tp = (plot_data["result"] == "TP").sum()
fp = (plot_data["result"] == "FP").sum()
tn = (plot_data["result"] == "TN").sum()
fn = (plot_data["result"] == "FN").sum()

table6 = pd.DataFrame(
    {
        "Predicted Dry": [
            tn,
            fn,
        ],
        "Predicted Wet": [
            fp,
            tp,
        ],
    },
    index=[
        "Actual Dry",
        "Actual Wet",
    ],
)

display(table6)

# --------------------------------------------------
# Save
# --------------------------------------------------
output_path = (
    DIR_EVAL /
    "table6_flood_detection_regression.csv"
)

table6.to_csv(output_path)

print(f"Saved: {output_path}")

# Table 7: Accuracy, Precision, Recall, F1-score for Grad-CAM localization (classification and regression)

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)

# --------------------------------------------------
# Controlled samples only
# --------------------------------------------------
plot_data = eval_summary[
    eval_summary["controlled"]
].copy()

# --------------------------------------------------
# Helper function
# --------------------------------------------------
def evaluate_model(plot_data, pred_wet):

    y_true = []
    y_pred = []

    for (_, row), pred in zip(plot_data.iterrows(), pred_wet):

        actual_wet = row["pair_id"].startswith("W")

        # Ground truth
        y_true.append(actual_wet)

        # Evaluation protocol
        if actual_wet:
            prediction = (
                pred
                and row["overlap"]
            )
        else:
            prediction = pred

        y_pred.append(prediction)

    return {
        "Accuracy": accuracy_score(
            y_true,
            y_pred,
        ),
        "Precision": precision_score(
            y_true,
            y_pred,
            zero_division=0,
        ),
        "Recall": recall_score(
            y_true,
            y_pred,
            zero_division=0,
        ),
        "F1 Score": f1_score(
            y_true,
            y_pred,
            zero_division=0,
        ),
    }

# --------------------------------------------------
# Classification model
# --------------------------------------------------
cls_metrics = evaluate_model(
    plot_data,
    plot_data["cls_prob"] >= THRESHOLD_CLS,
)

# --------------------------------------------------
# Regression model
# --------------------------------------------------
reg_metrics = evaluate_model(
    plot_data,
    plot_data["reg_pred"] >= THRESHOLD_REG,
)

# --------------------------------------------------
# Table 7
# --------------------------------------------------
table7 = pd.DataFrame(
    {
        "Classification": cls_metrics,
        "Regression": reg_metrics,
    }
).round(3)

display(table7)

# --------------------------------------------------
# Save
# --------------------------------------------------
output_path = (
    DIR_EVAL /
    "table7_flood_detection_metrics.csv"
)

table7.to_csv(output_path)

print(f"Saved: {output_path}")

In [ ]:
import rasterio
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# --------------------------------------------------
# Output directory
# --------------------------------------------------
output_dir = DIR_EVAL / "zscore_histograms"
output_dir.mkdir(exist_ok=True)

# --------------------------------------------------
# Non-controlled patches only
# --------------------------------------------------
plot_data = groups_patch_paths[
    ~groups_patch_paths["controlled"]
].copy()

# --------------------------------------------------
# Plot one histogram for each patch
# --------------------------------------------------
for _, row in plot_data.iterrows():

    pair_id = row["pair_id"]
    patch_id = row["patch_id"]

    with rasterio.open(row["zscore"]) as src:
        z_patch = src.read(1)

    # Remove NoData, NaN and Inf
    z = z_patch[np.isfinite(z_patch)]

    if src.nodata is not None:
        z = z[z != src.nodata]

    plt.figure(figsize=(5, 4))

    plt.hist(
        z,
        bins=50,
        range=(-4, 4),
    )

    plt.xlim(-4, 4)

    plt.title(f"{pair_id} - {patch_id}")
    plt.xlabel("Z-score")
    plt.ylabel("Pixel Count")

    plt.tight_layout()

    plt.savefig(
        output_dir / f"{pair_id}_{patch_id}.png",
        dpi=300,
    )

    plt.close()

print(f"Saved {len(plot_data)} histograms to {output_dir}")

In [ ]:
import rasterio
import matplotlib.pyplot as plt

In [ ]:
def read_raster(path):
    with rasterio.open(path) as src:
        return src.read(1)


def show_patch(
    groups_patch_paths,
    eval_summary,
    pair_id,
    patch_id,
    controlled,
    save=False,
    output_dir=".",
):
    row = groups_patch_paths.loc[
        (groups_patch_paths["pair_id"] == pair_id)
        & (groups_patch_paths["patch_id"] == patch_id)
        & (groups_patch_paths["controlled"] == controlled)
    ].iloc[0]

    stats = eval_summary.loc[
        (eval_summary["pair_id"] == pair_id)
        & (eval_summary["patch_id"] == patch_id)
        & (eval_summary["controlled"] == controlled)
    ].iloc[0]

    sar_base = read_raster(row["sar_base"])
    sar_target = read_raster(row["sar_target"])
    zscore = read_raster(row["zscore"])
    depmap = read_raster(row["depmap"])
    ndvi = read_raster(row["ndvi_target"])
    gradcam_cls = read_raster(row["gradcam_cls"])
    gradcam_reg = read_raster(row["gradcam_reg"])

    fig, axes = plt.subplots(1, 7, figsize=(18, 3.5))

    # SAR Base
    axes[0].imshow(sar_base, cmap="gray")
    axes[0].set_title("SAR Base")
    axes[0].axis("off")

    # SAR Target
    axes[1].imshow(sar_target, cmap="gray")
    axes[1].set_title("SAR Target")
    axes[1].axis("off")

    # Z-score
    im = axes[2].imshow(zscore, cmap="gray")
    axes[2].set_title("Z-score")
    axes[2].axis("off")
    plt.colorbar(im, ax=axes[2], fraction=0.046)

    # Depression
    im = axes[3].imshow(depmap, cmap="gray")
    axes[3].set_title("Depression")
    axes[3].axis("off")
    plt.colorbar(im, ax=axes[3], fraction=0.046)

    # NDVI
    im = axes[4].imshow(ndvi, cmap="gray")
    axes[4].set_title("NDVI")
    axes[4].axis("off")
    plt.colorbar(im, ax=axes[4], fraction=0.046)

    # Classification Grad-CAM
    axes[5].imshow(sar_target, cmap="gray")
    im = axes[5].imshow(
        gradcam_cls,
        cmap="jet",
        alpha=0.5,
        vmin=0,
        vmax=1,
    )
    axes[5].set_title(
        f"Classification\nP={stats['cls_prob']:.3f}"
    )
    axes[5].axis("off")
    plt.colorbar(im, ax=axes[5], fraction=0.046)

    # Regression Grad-CAM
    axes[6].imshow(sar_target, cmap="gray")
    im = axes[6].imshow(
        gradcam_reg,
        cmap="jet",
        alpha=0.5,
        vmin=0,
        vmax=1,
    )
    axes[6].set_title(
        f"Regression\nPred={stats['reg_pred']:.3f}"
    )
    axes[6].axis("off")
    plt.colorbar(im, ax=axes[6], fraction=0.046)

    fig.suptitle(patch_id, fontsize=14)
    plt.tight_layout()
    if save:
        output_dir = Path(output_dir)
        output_dir.mkdir(parents=True, exist_ok=True)

        condition = (
            "controlled"
            if controlled
            else "normal"
        )

        output_file = (
            output_dir /
            f"patches_{pair_id}_{patch_id}_{condition}.png"
        )

        plt.savefig(
            output_file,
            dpi=300,
            bbox_inches="tight",
        )

        print(f"Saved to {output_file}")
    else:
        plt.show()

    plt.close(fig)


In [ ]:
for _, row in groups_patch_paths.iterrows():

    condition = (
        "controlled"
        if row["controlled"]
        else "normal"
    )

    output_dir = (
        DIR_EVAL
        / "patches"
        / f"{row['pair_id']}-{condition}"
    )

    show_patch(
        groups_patch_paths=groups_patch_paths,
        eval_summary=eval_summary,
        pair_id=row["pair_id"],
        patch_id=row["patch_id"],
        controlled=row["controlled"],
        save=True,
        output_dir=output_dir,
    )

In [ ]:
patch_centers = pd.read_csv(FILEPATH_PATCH_CENTERS)
patches = patch_centers[patch_centers["usage"]=="test"]["id"].to_list()
patches

for patch in patches:
    show_patch(
        groups_patch_paths,
        eval_summary,
        patch,
        save=True,
        output_dir=DIR_EVAL / "patches"
    )